# Shared Linear Regression Baseline

Run Jupyter from the repository root. This example uses only the fixed temporal split and five shared baseline features.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from src.evaluation import evaluate
from src.features import BASELINE_FEATURES, make_features, make_preprocessor
from src.validation import load_data

## Load, clean, and split

`load_data` reads only `data/train.csv`, applies shared cleaning, and uses the fixed date ranges. It preserves original row order and index labels.

In [2]:
train_df, valid_df = load_data()

print(f"Training:   {len(train_df):,} rows, {train_df['RENT_APPROVAL_DATE'].min()} to {train_df['RENT_APPROVAL_DATE'].max()}")
print(f"Validation: {len(valid_df):,} rows, {valid_df['RENT_APPROVAL_DATE'].min()} to {valid_df['RENT_APPROVAL_DATE'].max()}")

Training:   117,988 rows, 2021-01 to 2024-04
Validation: 32,012 rows, 2024-05 to 2025-03


## Deterministic features

Feature construction below does not learn from data. Imputation, scaling, and encoding do learn from data, so they remain unfitted until the training Pipeline is fitted.

In [3]:
X_train = make_features(train_df)
X_valid = make_features(valid_df)
y_train = train_df['MONTHLY_RENT']
y_valid = valid_df['MONTHLY_RENT']

assert tuple(X_train.columns) == BASELINE_FEATURES
assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)
X_train.head()

,floor_area_sqm,flat_age,month_index,TOWN,FLAT_TYPE
0,93.0,9.0,0,punggol,4 room
1,110.0,20.0,0,sengkang,5 room
2,65.0,45.0,0,queenstown,3 room
3,96.0,11.0,0,sengkang,4 room
4,102.0,20.0,0,sembawang,4 room


## Fit and evaluate

The Pipeline fits the imputer, scaler, one-hot encoder, and regressor on training rows only. Replace the final model step to run a comparable model experiment.

In [4]:
experiment = Pipeline([
    ('preprocess', make_preprocessor()),
    ('model', LinearRegression()),
])

experiment.fit(X_train, y_train)
predictions = experiment.predict(X_valid)

metrics = evaluate(y_valid, predictions)
assert all(np.isfinite(value) for value in metrics.values())
metrics

{'RMSE': 616.2782613839569,
 'MAE': 460.26833788854356,
 'R2': 0.12010009732010707,
 'Mean Error': 351.261973376522}